In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/

In [ ]:
!pip install -q datasets

# 라이브러리

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy
from typing import Dict, List, Tuple, Optional, Literal, Iterable, Set
import dataclasses
import html

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# nltk, stopwords
import nltk
from nltk.corpus import stopwords

# Torch 모델 구조
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# BERT Model
# Fine Tuning
from transformers import AutoModel, AutoTokenizer
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.modeling_outputs import SequenceClassifierOutput

# 시각화
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
import textwrap
import networkx as nx
from IPython.display import HTML


# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

## Proposed Model

In [ ]:
class ReviewPairDataset(Dataset):
    def __init__(self, user_texts, item_texts, user_ids, item_ids, ratings, tokenizer, max_length=512):
        self.user_texts = user_texts
        self.item_texts = item_texts
        self.user_ids = user_ids
        self.item_ids = item_ids
        self.ratings = ratings
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.user_texts)

    def __getitem__(self, idx):
        user_enc = self.tokenizer(self.user_texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        item_enc = self.tokenizer(self.item_texts[idx], truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt")
        user_id = torch.tensor(self.user_ids[idx], dtype=torch.long)
        item_id = torch.tensor(self.item_ids[idx], dtype=torch.long)
        rating = torch.tensor(float(self.ratings[idx]), dtype=torch.float)

        return {
            "input_ids_user": user_enc["input_ids"].squeeze(0),
            "attention_mask_user": user_enc["attention_mask"].squeeze(0),
            "input_ids_item": item_enc["input_ids"].squeeze(0),
            "attention_mask_item": item_enc["attention_mask"].squeeze(0),
            "labels": rating,
            "user_id": user_id,
            "item_id": item_id
        }

In [ ]:
def make_mlp(sizes, activation=nn.ReLU):
    """
    중간 feature extractor용 MLP
    모든 Linear 뒤에 activation 함수가 붙음 (마지막 레이어 포함)
    """
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        layers.append(activation())
    return nn.Sequential(*layers)

In [ ]:
def make_last_mlp(sizes, activation=nn.ReLU):
    """
    최종 예측 MLP
    마지막 Linear 뒤에는 activation이 없음 (즉, 마지막은 Linear-only)
    """
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        if i < len(sizes) - 2:
            layers.append(activation())
    return nn.Sequential(*layers)

In [ ]:
class Co_Attention_Layer(nn.Module):
    def __init__(self, emb_dim=768, nhead=8):
        super().__init__()
        self.attn_user_to_item = nn.MultiheadAttention(embed_dim=emb_dim, num_heads=nhead, batch_first=True)
        self.attn_item_to_user = nn.MultiheadAttention(embed_dim=emb_dim, num_heads=nhead, batch_first=True)

    def forward(self, user_vec, item_vec):
        # 입력: [batch, emb]
        user_vec = user_vec.unsqueeze(1)  # [batch, 1, emb]
        item_vec = item_vec.unsqueeze(1)  # [batch, 1, emb]

        # ① user attends to item (Q=user, K/V=item)
        attn_u2i, _ = self.attn_user_to_item(query=user_vec, key=item_vec, value=item_vec)  # [batch, 1, emb]

        # ② item attends to user (Q=item, K/V=user)
        attn_i2u, _ = self.attn_item_to_user(query=item_vec, key=user_vec, value=user_vec)  # [batch, 1, emb]

        # Flatten and concat both directions
        concat = torch.cat([attn_u2i.squeeze(1), attn_i2u.squeeze(1)], dim=-1)  # [batch, emb*2]
        return concat

In [ ]:
class MAS_Rec(nn.Module):
    def __init__(self, n_users, n_items, user_model, item_model, nhead=8):
        super().__init__()

        # Interaction IDs
        self.user_id = nn.Embedding(n_users, 64)
        self.item_id = nn.Embedding(n_items, 64)

        # Two Tower Bert
        self.user_model = user_model
        self.item_model = item_model

        # [CLS] Dense
        self.cls_dense = make_mlp([768, 512, 256])

        # Co-Attention (bi-directional cross attention)
        # 256([CLS] dense output emb) + 64(interaction emb)
        self.cross_attn = Co_Attention_Layer(emb_dim=320, nhead=nhead)

        # Last Dense
        self.fc_final = make_last_mlp([640, 256, 1])

    def forward(self, user_id, item_id, input_ids_user, attention_mask_user, input_ids_item, attention_mask_item, labels=None):

        # Interaction IDs
        user_emb = self.user_id(user_id)    # [batch, 64]
        item_emb = self.item_id(item_id)    # [batch, 64]

        # Two Tower BERT
        # 여기에서 input_ids는 token_id(token index)
        user_cls = self.user_model(input_ids=input_ids_user, attention_mask=attention_mask_user).last_hidden_state[:, 0, :]
        item_cls = self.item_model(input_ids=input_ids_item, attention_mask=attention_mask_item).last_hidden_state[:, 0, :]

        # [CLS] Dense
        user_cls = self.cls_dense(user_cls)
        item_cls = self.cls_dense(item_cls)

        # Concat
        user_feat = torch.cat([user_emb, user_cls], dim=-1)
        item_feat = torch.cat([item_emb, item_cls], dim=-1)

        # Co-Attetnion
        co_attn_output = self.cross_attn(user_feat, item_feat)  # [batch, 512]

        # Last Dense Layer
        logits = self.fc_final(co_attn_output).squeeze(1)

        if labels is not None:
            loss = nn.MSELoss()(logits, labels)
            return {"loss": loss, "logits": logits}
        return {"logits": logits}

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions  # 그대로 사용!

    # 지표 계산
    mae = mean_absolute_error(labels, preds)
    mse = mean_squared_error(labels, preds)
    rmse = np.sqrt(mse)

    epsilon = 1e-8
    mape = np.mean(np.abs((labels - preds) / (labels + epsilon))) * 100

    return {
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "mape": mape
    }

## Explainable AI

In [ ]:
@dataclasses.dataclass
class InputBatch:
    user_id: torch.Tensor
    item_id: torch.Tensor
    user_input_ids: torch.Tensor
    user_attn_mask: torch.Tensor
    item_input_ids: torch.Tensor
    item_attn_mask: torch.Tensor

In [ ]:
class masrec_attention_rollout:
    def __init__(self, model: nn.Module, pad_id_user: int, pad_id_item: int, device: Optional[torch.device] = None):
        self.model = model
        self.model._pad_token_id_user = pad_id_user
        self.model._pad_token_id_item = pad_id_item
        self.device = device or next(model.parameters()).device

    # ---------- 공통 헬퍼 ----------
    @staticmethod
    def _fuse_heads(att: torch.Tensor, mode: str = "mean") -> torch.Tensor:
        # att: (B, H, L, L) -> (B, L, L)
        if mode == "mean":
            return att.mean(dim=1)
        elif mode == "sum":
            return att.sum(dim=1)
        elif mode == "max":
            return att.max(dim=1).values
        raise ValueError("head_fusion must be in {'mean','sum','max'}")

    @staticmethod
    def _col_mask(A: torch.Tensor, attn_mask: Optional[torch.Tensor]) -> torch.Tensor:
        if attn_mask is None: return A
        return A * attn_mask.unsqueeze(1).to(A.dtype)  # (B,1,L)

    @staticmethod
    def _row_mask(A: torch.Tensor, attn_mask: Optional[torch.Tensor]) -> torch.Tensor:
        if attn_mask is None: return A
        return A * attn_mask.unsqueeze(2).to(A.dtype)  # (B,L,1)

    @staticmethod
    def _add_residual_row_norm(A: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
        # A: (B,L,L)
        B, L, _ = A.shape
        eye = torch.eye(L, device=A.device, dtype=A.dtype).expand(B, L, L)
        A = A + eye
        A = A / (A.sum(dim=-1, keepdim=True) + eps)
        return A

    @torch.no_grad()
    def get_self_attn_rollout(
        self,
        model_bert: nn.Module,
        input_ids: torch.Tensor,
        attn_mask: torch.Tensor,
        *,
        head_fusion: str = "mean",
        add_residual: bool = True,
        mask_rows: bool = False,
        source_index: int | torch.Tensor = 0,     # int 또는 (B,) 텐서
        exclude_token_ids: Optional[list[int]] = None,
        return_full_matrix: bool = False,
        return_cum_mats: bool = False,            # ★ 누적 행렬 리스트 반환 옵션
        eps: float = 1e-6,
    ):
        """
        반환 형태:
          - return_cum_mats=False, return_full_matrix=False: (B, L)  # importance
          - return_cum_mats=False, return_full_matrix=True : (B, L, L)  # 최종 M
          - return_cum_mats=True: (importance or M), cum_mats(list of (B,L,L))
        """
        outputs = model_bert(
            input_ids=input_ids,
            attention_mask=attn_mask,
            output_attentions=True,
            return_dict=True,
        )
        attn_list = outputs.attentions  # tuple of (B,H,L,L)
        # 1) 헤드 결합
        fused = [self._fuse_heads(a, head_fusion) for a in attn_list]  # (B,L,L)

        # 2) 마스킹
        fused = [self._col_mask(A, attn_mask) for A in fused]
        if mask_rows:
            fused = [self._row_mask(A, attn_mask) for A in fused]

        # 3) 잔차 + 행정규화
        if add_residual:
            fused = [self._add_residual_row_norm(A, eps=eps) for A in fused]
        else:
            fused = [A / (A.sum(dim=-1, keepdim=True) + eps) for A in fused]

        # 4) 누적 곱
        M = fused[0]
        cum_mats: list[torch.Tensor] = [M] if return_cum_mats else None
        for i in range(1, len(fused)):
            M = torch.bmm(M, fused[i])
            if return_cum_mats:
                cum_mats.append(M)

        # 5) 반환 구성
        if return_full_matrix:
            result = M  # (B,L,L)
        else:
            # source_index가 배치별 상이한 경우 지원
            if isinstance(source_index, torch.Tensor):
                # source_index: (B,)
                B, L, _ = M.shape
                arange = torch.arange(B, device=M.device)
                result = M[arange, source_index, :]  # (B,L)
            else:
                result = M[:, source_index, :]       # (B,L)

            # exclude ids 제거 후 재정규화
            if exclude_token_ids is not None and len(exclude_token_ids) > 0:
                keep = torch.ones_like(input_ids, dtype=result.dtype)
                for tid in exclude_token_ids:
                    keep = keep * (input_ids != tid).to(result.dtype)
                result = result * keep
                result = result / (result.sum(dim=-1, keepdim=True) + eps)

        if return_cum_mats:
            return result, [cm.detach() for cm in cum_mats]  # torch 텐서 리스트
        return result


## VIsualization

### preprocessing

In [ ]:
def to_cpu_numpy(x: torch.Tensor) -> np.ndarray:
    return x.detach().float().cpu().numpy()

In [ ]:
# 시각화를 위해 불용어를 attention weight에서 제거
def build_stopwords(
    lang: str = "english",
    extra_stopwords: Optional[Iterable[str]] = None,
    keep_words: Optional[Iterable[str]] = None
) -> Set[str]:
    """
    NLTK 불용어 + 사용자 추가/보존 단어 반영
    """
    try:
        _ = stopwords.words(lang)
    except LookupError:
        nltk.download("stopwords")
    sw = set(stopwords.words(lang))

    if extra_stopwords:
        sw |= set(w.lower() for w in extra_stopwords)
    if keep_words:
        sw -= set(w.lower() for w in keep_words)
    return sw

In [ ]:
def topk_indices(arr: np.ndarray, k: int = 10) -> np.ndarray:
    k = min(k, arr.size)
    return np.argsort(arr)[::-1][:k]

In [ ]:
def normalize_vec(x: np.ndarray) -> np.ndarray:
    """합이 정확히 1.0이 되도록 정규화 (합이 0이면 그대로 0들 반환)"""
    x = np.asarray(x, dtype=np.float64)
    s = x.sum()
    if s <= 0:
        return np.zeros_like(x, dtype=np.float32)
    return (x / s).astype(np.float32)

In [ ]:
def filter_tokens(
    tokens: Iterable[str],
    scores: np.ndarray,
    exclude_tokens: Optional[Iterable[str]] = None,
    drop_punct: bool = True,
    punct_tokens: Optional[Iterable[str]] = None,
    drop_stopwords: bool = False,
    stopwords: Optional[Iterable[str]] = None,
    case_insensitive: bool = True,
) -> Tuple[List[str], np.ndarray]:
    """
    토큰/스코어 1차 필터링:
    - 스페셜 토큰 제거
    - 구두점: 표시용으로는 남기되, 스코어는 0 처리
    - 불용어 제거(옵션)
    """
    exclude = set(exclude_tokens or [])
    punct = set(punct_tokens or [])
    kept_toks, kept_scores = [], []
    for t, s in zip(tokens, scores):
        if t in exclude:
            continue
        if drop_punct and t in punct:
            kept_toks.append(t)
            kept_scores.append(0.0)   # 표시만 하고 중요도는 0
        else:
            kept_toks.append(t)
            kept_scores.append(float(s))
    return kept_toks, np.array(kept_scores, dtype=np.float32)

In [ ]:
def merge_wordpieces(
    tokens: List[str],
    scores: np.ndarray,
    aggregate: str = "sum",
) -> Tuple[List[str], np.ndarray]:
    """
    BERT WordPiece 병합 → 단어 단위 중요도.
    aggregate: 'sum' | 'mean' (서브워드 집계 방식)
    """
    words, accs = [], []
    buf, acc, cnt = "", 0.0, 0

    for t, s in zip(tokens, scores):
        if t.startswith("##"):
            # 서브워드 이어붙이기
            buf += t[2:]
            acc += float(s)
            cnt += 1
        else:
            # 새로운 단어 시작 -> 기존 flush
            if buf:
                accs.append(acc if aggregate == "sum" else (acc / max(cnt, 1)))
                words.append(buf)
            buf, acc, cnt = t, float(s), 1

    if buf:
        accs.append(acc if aggregate == "sum" else (acc / max(cnt, 1)))
        words.append(buf)

    accs = np.maximum(np.array(accs, dtype=np.float32), 0.0)  # 음수 방지 # 정규화는 나중(불용어 0 마스킹 후) 수행

    return words, accs

In [ ]:
# 불용어 중요도 0으로 마스킹 + 재정규화
# token은 그대로 표시

def mask_stopwords_to_zero(
    words: List[str],
    word_scores: np.ndarray,
    stopwords: Iterable[str],
    case_insensitive: bool = True,
    soft_mask_factor: float = 0.0,  # 0.0이면 완전 0, 0<α<1이면 약하게 남김
):
    sw = {w.lower() for w in stopwords}
    masked = word_scores.copy()
    for i, w in enumerate(words):
        key = w.lower() if case_insensitive else w
        if key in sw:
            masked[i] = masked[i] * soft_mask_factor  # 기본 0.0
    # 비-불용어 합으로 재정규화 (불용어는 0 그대로)
    pos_sum = masked.sum()
    if pos_sum > 0:
        masked = masked / pos_sum
    # 모두 불용어였던 극단적 케이스는 그대로 반환(합=0)
    return masked

In [ ]:
# 온점, 구두점, 따옴표 등등 시각화에 방해되는 부분을 제거
SPECIAL_TOKENS_DEFAULT = {"[PAD]", "[CLS]", "[SEP]"}
PUNCT_TOKENS_DEFAULT = {
    ".", ",", ";", "!", "?", ":", "-", "—", "–",
    "''", "``", "'", '"', "(", ")", "[", "]", "{", "}", "...", "…",
    "/", "\\", "|", "·", "`"
}

DISPLAY_PUNCT_WHITELIST = {".", "?", "!"}
DISPLAY_PUNCT_HIDE = {
    ",", ";", ":", "-", "—", "–",
    "''", "``", "'", '"', "(", ")", "[", "]", "{", "}", "...", "…",
    "/", "\\", "|", "·", "`"
}

def apply_display_visibility(
    toks: List[str],
    scores: np.ndarray,
    hide_set: Iterable[str] = DISPLAY_PUNCT_HIDE,
    whitelist: Iterable[str] = DISPLAY_PUNCT_WHITELIST,
) -> Tuple[List[str], np.ndarray]:
    """
    렌더링 전 단계에서 '표시만' 거르기.
    - 화이트리스트에 없는 구두점은 화면에서 숨김
    - 단어/서브워드는 모두 표시
    """
    hide = set(hide_set)
    white = set(whitelist)

    vis_toks, vis_scores = [], []
    for t, s in zip(toks, scores):
        # 구두점이면: 화이트리스트만 통과, 그 외는 숨김
        if t in hide and t not in white:
            continue
        vis_toks.append(t)
        vis_scores.append(s)
    return vis_toks, np.asarray(vis_scores, dtype=np.float32)


In [ ]:
def tokens_to_word_importance(
    tokens: List[str],
    token_importance: np.ndarray,
    stopwords: Iterable[str],
    exclude_special: Iterable[str] = SPECIAL_TOKENS_DEFAULT,
    drop_punct: bool = True,
    punct_tokens: Iterable[str] = PUNCT_TOKENS_DEFAULT,
    aggregate: str = "sum",
    soft_mask_factor: float = 0.0,  # 필요시 0.1으로 완전 제거하지 않고 약하게 표시
    hide_punct_for_display: bool = True,
) -> Tuple[List[str], np.ndarray, List[str], np.ndarray]:
    """
    1) 스페셜/구두점/불용어 필터 → 2) 워드피스 병합 → 3) 정규화
    반환: (단어 리스트, 단어 중요도; 합=1.0)
    """
    # 스페셜/구두점만 제거(불용어는 유지)
    ftoks, fscores = filter_tokens(
        tokens, token_importance,
        exclude_tokens=exclude_special,
        drop_punct=drop_punct,
        punct_tokens=punct_tokens,
    )
    if len(ftoks) == 0:
        z = np.array([], dtype=np.float32)
        return [], z, [], z

    # 워드피스 병합
    words, word_scores = merge_wordpieces(ftoks, fscores, aggregate=aggregate)

    # 불용어 중요도만 0(또는 small)으로 → 비-불용어 합으로 정규화
    masked_scores = mask_stopwords_to_zero(
        words, word_scores, stopwords,
        case_insensitive=True,
        soft_mask_factor=soft_mask_factor,
    )

    # 4) 표시용: 구두점 중 , - : ; 따옴표 등은 숨김
    if hide_punct_for_display:
        disp_words, disp_scores = apply_display_visibility(words, masked_scores)
        return words, masked_scores, disp_words, disp_scores
    else:
        return words, masked_scores

### topk plot

In [ ]:
def plot_topk_bar(tokens: List[str], importance: np.ndarray, k: int = 10,
                  title: str = "Top-k token importance", colors: str = None):
    idx = topk_indices(importance, k=k)
    labels = [tokens[i] for i in idx]
    vals = importance[idx]
    plt.figure(figsize=(24, 6), dpi=160)
    plt.bar(range(len(idx)), vals, color=colors)
    plt.xticks(range(len(idx)), labels, rotation=45, ha='right', fontsize=15)
    plt.title(title, fontsize=15)
    plt.tight_layout()
    plt.show()

### heatmap plot

In [ ]:
def plot_full_heatmap(tokens: List[str], importance: np.ndarray, title: str = "Token importance heatmap"):
    """단일 벡터(행:1, 열:토큰) 히트맵. rollout M에서 [CLS]행만 쓴 경우 시각화에 적합."""
    plt.figure(figsize=(max(24, len(tokens) * 0.3), 2.5))
    mat = importance[None, :]  # (1, L)
    plt.imshow(mat, aspect='auto')
    plt.yticks([0], ["[CLS]→tokens"], fontsize=15)
    plt.xticks(range(len(tokens)), tokens, rotation=45, ha='right', fontsize=15)
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.show()

### word highlight html

In [ ]:
# ---------- 2) 보고서: WordPiece 병합 텍스트 하이라이트(HTML) ----------
# 파랑: rgba(0,0,255,{alpha})
# 초록: rgba(0,200,0,{alpha})
# 보라: rgba(128,0,128,{alpha})


def html_highlight_wrapped(tokens, importance, span_alpha_scale=1.0, mode="minmax",
                           max_width_px=900, font_size_px=14, line_height=1.5,
                           base_color=(0,0,255),
                           top_percent=0.10,     # ✅ 상위 N% Bold (0.10 = 상위 10%)
                           top_k=None,           # ✅ 상위 K개 Bold (top_k가 우선)
                           tie_mode="percentile", # {"percentile","cut"}: 동점 처리 방식
                           font_family="Times New Roman"
                           ):
    """
    WordPiece 병합 텍스트 하이라이트 (HTML)
    - tokens: 단어/토큰 리스트
    - importance: 중요도 배열
    - base_color: (R,G,B) 튜플, 예: (0,0,255)=파랑, (0,200,0)=초록, (128,0,128)=보라
    """
    # 1) 중요도 정규화
    imp = np.asarray(importance, dtype=np.float32)
    if imp.size == 0:
        return "<div></div>"
    if mode == "minmax":
        rng = imp.max() - imp.min()
        imp = (imp - imp.min()) / (rng + 1e-8)
    elif mode == "softmax":
        e = np.exp(imp - imp.max())
        imp = e / (e.sum() + 1e-8)

    words, scores = tokens, imp

    # 2) Bold 대상 인덱스 계산
    n = len(words)
    if top_k is not None:
        k = max(1, min(n, int(top_k)))
    else:
        k = max(1, int(np.ceil(n * float(top_percent))))

    if tie_mode == "percentile":
        # k개에 해당하는 퍼센타일 임계값 (상위 k개 위치의 점수에 해당)
        if k >= n:
            bold_mask = np.ones(n, dtype=bool)
        else:
            thresh = np.partition(scores, n - k)[n - k]  # 상위 k번째 값
            bold_mask = scores >= thresh                 # 동점은 모두 포함
    else:  # "cut": 정확히 k개만
        order = np.argsort(scores)[::-1]
        top_idx = set(order[:k])
        bold_mask = np.array([i in top_idx for i in range(n)], dtype=bool)

    # 3) 컨테이너 스타일
    head = f"""
<div style="
  max-width:{max_width_px}px;
  font-size:{font_size_px}px;
  line-height:{line_height};
  font-family:{font_family}, serif;
  white-space:normal; word-wrap:break-word; overflow-wrap:anywhere;
  letter-spacing:0.3px; word-spacing:2px;
  text-align:justify; text-justify:inter-word;
">
"""
    tails = "</div>"

    # 4) 토큰 span 생성 (Bold 적용)
    r, g, b = base_color
    spans = []
    for i, (w, a) in enumerate(zip(words, scores)):
        alpha = max(0.05, min(1.0, float(a) * span_alpha_scale))
        weight_style = "font-weight:700;" if bold_mask[i] else "font-weight:400;"
        bg = f"rgba({r},{g},{b},{alpha})"
        spans.append(
            f'<span style="background:{bg}; padding:0 3px; border-radius:3px; '
            f'margin:0 1px 2px 0; display:inline-block; vertical-align:middle; {weight_style}">'
            f'{html.escape(w)}</span>'
        )

    return head + " ".join(spans) + tails

### cumulative rollout plot

In [ ]:
# ---------- 3) 연구 진단: 누적 레이어별 히트맵/라인업 ----------

def per_layer_rollout(att_stack: List[torch.Tensor]) -> List[np.ndarray]:
    """
    att_stack: 레이어별 (B,H,L,L) 또는 (B,L,L) 텐서들의 리스트
    - H 차원이 있으면 평균으로 머지
    반환: 누적 rollout 행렬 M_t (t=1..L) 의 리스트 (각 (L,L), B=1 가정)
    """
    mats = []
    for a in att_stack:
        A = a
        if A.dim() == 4:  # (B,H,L,L)
            A = A.mean(dim=1)
        # (B,L,L) 가정
        A = A / (A.sum(dim=-1, keepdim=True) + 1e-6)
        mats.append(A)
    # 누적 곱
    cum = mats[0]
    outs = [to_cpu_numpy(cum[0])]
    for i in range(1, len(mats)):
        cum = torch.bmm(cum, mats[i])
        outs.append(to_cpu_numpy(cum[0]))
    return outs  # List of (L,L)

In [ ]:
def plot_rollout_lineup(cum_mats: List[np.ndarray], tokens: List[str], source_idx: int = 0, max_cols: int = 4):
    """
    레이어가 증가할수록 [source] 행의 분포가 어떻게 변하는지 라인업(여러 히트맵)으로 보여줌.
    """
    T = len(cum_mats)
    rows = int(np.ceil(T / max_cols))
    cols = min(T, max_cols)
    L = len(tokens)

    fig_w = max(6, cols * (L * 0.25))  # 토큰 수 비례
    fig_h = max(3, rows * 2.5)

    plt.figure(figsize=(fig_w, fig_h))
    for i, M in enumerate(cum_mats):
        plt.subplot(rows, cols, i+1)
        row = M[source_idx:source_idx+1, :]
        plt.imshow(row, aspect='auto')
        plt.title(f"Layer {i+1}")
        plt.xticks(range(L), tokens, rotation=90, fontsize=8)  # 글꼴 줄이기
        plt.yticks([0], [f"src={tokens[source_idx]}"])
    plt.tight_layout()
    plt.show()

In [ ]:
# ---------- 5) 추천 설명: 유저/아이템 비교 + 교집합 ----------

def compare_user_item(
    user_tokens: List[str], user_imp: np.ndarray,
    item_tokens: List[str], item_imp: np.ndarray,
    k: int = 10, title_prefix: str = "User vs Item"
):
    # 빠른 bar 비교
    ui = topk_indices(user_imp, k)
    ii = topk_indices(item_imp, k)
    ut = [user_tokens[i] for i in ui]; uv = user_imp[ui]
    it = [item_tokens[i] for i in ii]; iv = item_imp[ii]

    plt.figure(figsize=(6, 3))
    plt.bar(range(len(ui)), uv)
    plt.xticks(range(len(ui)), ut, rotation=45, ha='right')
    plt.title(f"{title_prefix}: User Top-{len(ui)}")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 3))
    plt.bar(range(len(ii)), iv)
    plt.xticks(range(len(ii)), it, rotation=45, ha='right')
    plt.title(f"{title_prefix}: Item Top-{len(ii)}")
    plt.tight_layout()
    plt.show()

    # 교집합(단어 기준; WordPiece 병합 후 쓰면 더 선명)
    U = set(ut)
    I = set(it)
    inter = sorted(U & I)
    only_u = sorted(U - I)
    only_i = sorted(I - U)

    print(f"[공통 키워드] {inter}")
    print(f"[User 전용] {only_u}")
    print(f"[Item 전용] {only_i}")

# Load Dataset

In [ ]:
# 데이터 로딩
df = pd.read_pickle('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_aspect_final.pkl')
print(df.shape)
df.head()

# Prepare Input

In [ ]:
# Encode user and item IDs
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

# fit-transform
df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['asin'] = item_encoder.fit_transform(df['asin'])

# 실험 데이터의 고유한 user와 item 수를 집계
n_users = df['user_id'].nunique()
n_items = df['asin'].nunique()

Num_User = n_users + 1
Num_Item = n_items + 1

In [ ]:
# train, valid, test으로 데이터셋 분리
train, test = train_test_split(df, test_size=0.2, random_state=42)
train, valid = train_test_split(train, test_size=0.125, random_state=42)

train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)
test = test.reset_index(drop=True)

In [ ]:
train.shape, valid.shape, test.shape

In [ ]:
n_users, n_items

In [ ]:
# train, test, valid 변수 선언
train_user_id = train['user_id'].to_numpy()
train_item_id = train['asin'].to_numpy()
train_summary = train['LLaMA_Output']
train_image_description = train['LLaVA_Output']
train_rating = train['rating'].to_numpy()

valid_user_id = valid['user_id'].to_numpy()
valid_item_id = valid['asin'].to_numpy()
valid_summary = valid['LLaMA_Output']
valid_image_description = valid['LLaVA_Output']
valid_rating = valid['rating'].to_numpy()

test_user_id = test['user_id'].to_numpy()
test_item_id = test['asin'].to_numpy()
test_summary = test['LLaMA_Output']
test_image_description = test['LLaVA_Output']
test_rating = test['rating'].to_numpy()

# Load Inference Model

## Two Tower BERT

```python
MAS_Rec(
  user_id: Embedding(num_users, 64)
  item_id: Embedding(num_items, 64)
  user_model: BertModel
  item_model: BertModel
  cls_dense: MLP(768→512→256)
  co_attn: Co_Attention_Layer(MultiheadAttention*2, emb_dim=320)
  fc_final: MLP(640→256→1)
)

```

# Attention Rollout
- Quantifying Attention Flow in Transformers
- Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics (ACL)
- 1219회 인용
- https://arxiv.org/abs/2005.00928

In [ ]:
# BERT 모델 Loading

# 사용자 텍스트 처리 모델
BERT_review = AutoModel.from_pretrained("bert-base-uncased", attn_implementation="eager")

# 아이템 텍스트 처리 모델
BERT_image = AutoModel.from_pretrained("bert-base-uncased", attn_implementation="eager")

# tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", attn_implementation="eager")

In [ ]:
# 1) 모델 객체 먼저 생성 (구조 동일해야 함)
model = MAS_Rec(Num_User, Num_Item, user_model=BERT_review, item_model=BERT_image, nhead=8)

# 2) 가중치 로드
model.load_state_dict(torch.load("/content/drive/MyDrive/MAS_Rec/MAS_Rec.pth", map_location="cuda"))

# 3) 토크나이저
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/MAS_Rec/MAS_Rec_Tokenizer")

In [ ]:
for name, module in model.named_children():
    print(name, module)

In [ ]:
pad_id = tokenizer.pad_token_id
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
pad_id

## Prepare Inference Data

In [ ]:
test_candidate = test.loc[test['LLaMA_Output'].apply(lambda row: len(row)) <= 600, ['title_y', 'LLaMA_Output', 'LLaVA_Output', 'large_image_url']]
test_candidate = test_candidate.loc[test_candidate['LLaVA_Output'].apply(lambda row: len(row)) <= 550]
test_candidate

In [ ]:
train.loc[train['title_y'].str.contains('Hanna Nikole'), 'rating'].mean()

In [ ]:
sample_data = test[9362:9363].reset_index(drop=True)

sample_user_id = sample_data['user_id'].to_numpy()
sample_item_id = sample_data['asin'].to_numpy()
sample_summary = sample_data['LLaMA_Output']
sample_image_description = sample_data['LLaVA_Output']
sample_rating = sample_data['rating'].to_numpy()

In [ ]:
sample_dataset = ReviewPairDataset(sample_summary, sample_image_description, sample_user_id, sample_item_id, sample_rating, tokenizer, max_length=512)

In [ ]:
sample_data['medium_image_url'][0]

In [ ]:
# 1) 모델 래퍼(호출 가능)
model.eval()
model.to(device)

test_loader = DataLoader(sample_dataset, batch_size=1, shuffle=False)  # 👈 Dataset이 아니라 DataLoader!
batch_raw = next(iter(test_loader))

for k, v in batch_raw.items():
    if isinstance(v, torch.Tensor):
        batch_raw[k] = v.to(device)

ib = InputBatch(
    user_id=batch_raw["user_id"],                        # [1]
    item_id=batch_raw["item_id"],                        # [1]
    user_input_ids=batch_raw["input_ids_user"],          # [1, L_u]
    user_attn_mask=batch_raw["attention_mask_user"],     # [1, L_u]
    item_input_ids=batch_raw["input_ids_item"],          # [1, L_i]
    item_attn_mask=batch_raw["attention_mask_item"],     # [1, L_i]
)


with torch.no_grad():
    base_out = model(
        ib.user_id, ib.item_id,
        ib.user_input_ids, ib.user_attn_mask,
        ib.item_input_ids, ib.item_attn_mask
    )["logits"]
print(base_out)

In [ ]:
sample_data['title_y'][0]

In [ ]:
sample_data['rating'][0]

In [ ]:
sample_data['text'][0]

In [ ]:
sample_data['medium_image_url'][0]

In [ ]:
sample_data['LLaMA_Output'][0]

In [ ]:
sample_data['LLaVA_Output'][0]

## User Visualization

In [ ]:
# 2) XAI 유틸(호출 불가: 분석용 도우미)
xai_util = masrec_attention_rollout(model, pad_id_user=pad_id, pad_id_item=pad_id, device=device)   # model은 학습 완료된 원본, # user, item id 없어도 됨. 시각화용

In [ ]:
# attention weight 산출에서 제외할 불용어 선별
stop_words = build_stopwords(
    lang="english",
    extra_stopwords={"would", "could"},   # 필요시 추가
    keep_words={"not", "no"}              # 유지하고 싶은 단어
)

In [ ]:
# 예: 유저 인코더에서 기준 중요도, PAD/CLS/SEP 제외
exclude_ids_user = [pad_id]

importance_t, cum_mats_t = xai_util.get_self_attn_rollout(
    model_bert=model.user_model,
    input_ids=ib.user_input_ids,
    attn_mask=ib.user_attn_mask,
    head_fusion="mean",
    add_residual=True,
    mask_rows=False,
    source_index=0,                      # [CLS]
    exclude_token_ids=exclude_ids_user,
    return_full_matrix=False,
    return_cum_mats=True,               # ★ 누적 행렬 리스트 함께 받기
)

In [ ]:
# 시각화할 배치 인덱스
b_idx = 0

# 1) 텐서를 numpy로
importance = importance_t[b_idx].detach().float().cpu().numpy()   # (L,)
ids_user = ib.user_input_ids[b_idx].tolist()
tokens_all = tokenizer.convert_ids_to_tokens(ids_user)

# 2) 유효 길이로 자르기 (패딩 제거)
valid_len = int(ib.user_attn_mask[b_idx].sum().item())
tokens_v = tokens_all[:valid_len]
importance_v = importance[:valid_len]

# 3) attention rollout 결과를 numpy로 변환
importance = importance_t[b_idx].detach().float().cpu().numpy()           # (L,)
cum_mats = [cm[b_idx].detach().float().cpu().numpy() for cm in cum_mats_t]  # list of (L,L)

# 3) 스페셜/구두점 제거 → 병합 → 정규화
words, word_scores, disp_words, disp_scores = tokens_to_word_importance(
    tokens=tokens_v,
    token_importance=importance_v,
    stopwords=stop_words,
    drop_punct=True,
    aggregate="sum",
    soft_mask_factor=0.0,        # = 불용어 중요도 완전 0
    hide_punct_for_display=True,
)

# 검증
assert (len(words) == len(word_scores)) and (np.isclose(word_scores.sum(), 1.0) or len(word_scores)==0)

In [ ]:
# attention rollout weight 중 상위 50개를 시각화
# "red", "blue", "green", "orange", "purple", "pink", "cyan", "magenta", "yellow", "black", "white", "gray"
plot_topk_bar(words, word_scores, k=40, title="Top-k (CLS→token) importance (cleaned)", colors="orange")

In [ ]:
# weight 전체 heatmap
# plot_full_heatmap(words, word_scores, title="Importance heatmap")

In [ ]:
# WordPiece 병합 후 단어 단위 하이라이트

# 색 이름     | RGB 튜플          | 비고
# ------------|-------------------|----------------
# 밝은 빨강    | (255, 102, 102)  | 연한 레드
# 밝은 주황    | (255, 178, 102)  | 오렌지 느낌
# 밝은 노랑    | (255, 255, 153)  | 파스텔 옐로
# 밝은 연두    | (153, 255, 153)  | 라이트 그린
# 밝은 민트    | (153, 255, 204)  | 청록 계열
# 밝은 하늘    | (153, 204, 255)  | 연한 블루
# 밝은 파랑    | (102, 178, 255)  | 라이트 블루
# 밝은 보라    | (204, 153, 255)  | 라벤더 느낌
# 밝은 분홍    | (255, 153, 204)  | 핑크
# 밝은 회색    | (224, 224, 224)  | 배경용 무난


html_str = html_highlight_wrapped(disp_words, disp_scores, span_alpha_scale=1, mode="minmax",
                                  max_width_px = 400, font_size_px = 15, line_height = 1.5, base_color=(255, 178, 102), top_percent=0.20, top_k=None, tie_mode="percentile")

# html_str = html_highlight_wrapped(disp_words, disp_scores, span_alpha_scale=1.1, mode="minmax",
#                                   max_width_px = 800, font_size_px = 15, line_height = 1.5, base_color=(255, 178, 102), top_k=15, tie_mode="cut")

HTML(html_str)

## Item Visualization

In [ ]:
# 예: 아이템 인코더에서 기준 중요도, PAD/CLS/SEP 제외
exclude_ids_item = [pad_id]

importance_t, cum_mats_t = xai_util.get_self_attn_rollout(
    model_bert=model.item_model,
    input_ids=ib.item_input_ids,
    attn_mask=ib.item_attn_mask,
    head_fusion="mean",
    add_residual=True,
    mask_rows=False,
    source_index=0,                      # [CLS]
    exclude_token_ids=exclude_ids_item,
    return_full_matrix=False,
    return_cum_mats=True,               # ★ 누적 행렬 리스트 함께 받기
)

In [ ]:
# 시각화할 배치 인덱스
b_idx = 0

# 1) 텐서를 numpy로
importance = importance_t[b_idx].detach().float().cpu().numpy()   # (L,)
ids_item = ib.item_input_ids[b_idx].tolist()
tokens_all = tokenizer.convert_ids_to_tokens(ids_item)

# 2) 유효 길이로 자르기 (패딩 제거)
valid_len = int(ib.item_attn_mask[b_idx].sum().item())
tokens_v = tokens_all[:valid_len]
importance_v = importance[:valid_len]

# 3) attention rollout 결과를 numpy로 변환
importance = importance_t[b_idx].detach().float().cpu().numpy()           # (L,)
cum_mats = [cm[b_idx].detach().float().cpu().numpy() for cm in cum_mats_t]  # list of (L,L)

# 3) 스페셜/구두점 제거 → 병합 → 정규화
words, word_scores, disp_words, disp_scores = tokens_to_word_importance(
    tokens=tokens_v,
    token_importance=importance_v,
    stopwords=stop_words,
    drop_punct=True,
    aggregate="sum",
    soft_mask_factor=0.0,        # = 불용어 중요도 완전 0
    hide_punct_for_display=True,
)


# 검증
assert (len(words) == len(word_scores)) and (np.isclose(word_scores.sum(), 1.0) or len(word_scores)==0)

In [ ]:
# attention rollout weight 중 상위 50개를 시각화
plot_topk_bar(words, word_scores, k=30, title="Top-k (CLS→token) importance (cleaned)", colors="green")

In [ ]:
# weight 전체 heatmap
# plot_full_heatmap(words, word_scores, title="Importance heatmap")

In [ ]:
# WordPiece 병합 후 단어 단위 하이라이트
# - base_color: (R,G,B) 튜플, 예: (0,0,255)=파랑, (0,200,0)=초록, (128,0,128)=보라

# 색 이름     | RGB 튜플          | 비고
# ------------|-------------------|----------------
# 밝은 빨강    | (255, 102, 102)  | 연한 레드
# 밝은 주황    | (255, 178, 102)  | 오렌지 느낌
# 밝은 노랑    | (255, 255, 153)  | 파스텔 옐로
# 밝은 연두    | (153, 255, 153)  | 라이트 그린
# 밝은 민트    | (153, 255, 204)  | 청록 계열
# 밝은 하늘    | (153, 204, 255)  | 연한 블루
# 밝은 파랑    | (102, 178, 255)  | 라이트 블루
# 밝은 보라    | (204, 153, 255)  | 라벤더 느낌
# 밝은 분홍    | (255, 153, 204)  | 핑크
# 밝은 회색    | (224, 224, 224)  | 배경용 무난

html_str = html_highlight_wrapped(disp_words, disp_scores, span_alpha_scale=20, mode="minmax",
                                  max_width_px = 400, font_size_px = 15, line_height = 1.5, base_color=(153, 204, 255), top_percent=0.20, top_k=None, tie_mode="percentile")

# html_str = html_highlight_wrapped(disp_words, disp_scores, span_alpha_scale=5, mode="minmax",
#                                   max_width_px = 800, font_size_px = 15, line_height = 1.5, base_color=(153, 204, 255), top_k=20, tie_mode="cut")

HTML(html_str)

In [ ]:
# # # 토큰 개수 계산 함수 정의
# def count_tokens(text):
#     return len(tokenizer.encode(text, add_special_tokens=False))

# # 컬럼 추가
# test['num_tokens_llama'] = test['LLaMA_Output'].progress_apply(count_tokens)